In [1]:
# %%
import os
from dotenv import load_dotenv
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

load_dotenv()
AZURE_OPENAI_EMB_ENDPOINT = os.getenv("AZURE_OPENAI_EMB_ENDPOINT")
AZURE_OPENAI_EMB_API_KEY = os.getenv("AZURE_OPENAI_EMB_API_KEY")
AZURE_OPENAI_EMB_API_VERSION = os.getenv("AZURE_OPENAI_EMB_API_VERSION")
AZURE_OPENAI_EMB_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMB_DEPLOYMENT")

In [2]:
# %%
def get_embedding(text: str):
    url = f"{AZURE_OPENAI_EMB_ENDPOINT}/openai/deployments/{AZURE_OPENAI_EMB_DEPLOYMENT}/embeddings?api-version={AZURE_OPENAI_EMB_API_VERSION}"

    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_EMB_API_KEY
    }

    payload = {
        "input": text
    }

    response = requests.post(url, headers=headers, json=payload, verify=False)

    if response.status_code != 200:
        raise Exception(f"Error: {response.status_code}, {response.text}")

    result = response.json()

    # Extract embedding vector
    embedding = result["data"][0]["embedding"]

    return embedding


# Example usage
text = "Unlike natural language processing and computer vision, the development of Foun dation Models (FMs) for time series forecasting is blocked due to data scarcity."
vector = get_embedding(text)

print(type(vector))   # list
print(len(vector))    # dimension (e.g., 1536 or 3072 depending on model)
print(vector[:10])    # first 10 values



<class 'list'>
1536
[-0.018623933, -0.01875928, -0.0009558977, -0.008865316, -0.011511377, -0.004432658, -0.014766505, 0.019611975, -0.006990742, -0.03676061]


In [1]:
# %%
# -------------------------------
# 0. SET PROJECT ROOT (IMPORTANT)
# -------------------------------
import sys
import os
from weaviate.classes.config import Configure, Property
from weaviate.classes.data import DataObject

# PROJECT_ROOT = r"C:\Users\Jevinkumar.Palabhai\OneDrive - TVS Motor Company Ltd\Code-Space\G-RAG\proj-grag"

# PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
# sys.path.append(PROJECT_ROOT)
# if PROJECT_ROOT not in sys.path:



# -------------------------------
# 1. IMPORT MODULES
# -------------------------------
from services.ingestion.pdf_parser import extract_pages
from services.ingestion.block_extractor import extract_blocks
from services.ingestion.chunker import chunk_blocks, build_chunk_text

from services.embedding.embeddings import get_embedding
from services.graph.graph_pipeline import build_graph_from_chunks


# -------------------------------
# 2. INPUT
# -------------------------------
# GraphRAG_PAPER_PDF_PATH = r"C:\Users\Jevinkumar.Palabhai\Downloads\mini-GraphRAG.pdf"

# "C:\Users\Jevinkumar.Palabhai\Downloads\Federated Spatio-Temporal Graph Learning Method for Traffic Flow Forecasting.pdf"
# PDF_PATH = r"C:\Users\Dell\OneDrive - Indian Institute of Science\Academics\LAB\ArjunanSir-CPSdep\Federated Foundation Models on Heterogeneous Time Series (FFTS).pdf"
PDF_PATH = r"C:\Users\Dell\OneDrive - Indian Institute of Science\Academics\LAB\ArjunanSir-CPSdep\FeDaL.pdf"

# -------------------------------
# 3. INGESTION
# -------------------------------
print("Step 1: Extracting pages...")
GraphRAG_PAPER_pages = extract_pages(PDF_PATH)

print("Total pages:", len(GraphRAG_PAPER_pages))


print("\nStep 2: Extracting blocks...")
GraphRAG_PAPER_blocks = extract_blocks(GraphRAG_PAPER_pages)

print("Total blocks:", len(GraphRAG_PAPER_blocks))


print("\nStep 3: Chunking...")
GraphRAG_PAPER_chunks = chunk_blocks(GraphRAG_PAPER_blocks, max_tokens=300, overlap_tokens=50)

print("Total chunks:", len(GraphRAG_PAPER_chunks))


# -------------------------------
# 4. BUILD CHUNK TEXT
# -------------------------------
print("\nStep 4: Building chunk texts...")
GraphRAG_PAPER_chunk_texts = []

for i, chunk in enumerate(GraphRAG_PAPER_chunks):
    text = build_chunk_text(chunk)

    # Safety check (important)
    if text.strip():
        GraphRAG_PAPER_chunk_texts.append(text)

print("Valid chunk_texts:", len(GraphRAG_PAPER_chunk_texts))


# -------------------------------
# 5. GENERATE EMBEDDINGS
# -------------------------------
from services.vector_db.weaviate_client import WeaviateDB
import uuid

db = WeaviateDB()

print("Creating schema...")
db.create_schema()

print("Storing embeddings...")

chunk_ids = []
failed_chunks = []

for i, text in enumerate(GraphRAG_PAPER_chunk_texts):
    try:
        emb = get_embedding(text)

        cid = str(uuid.uuid4())
        chunk_ids.append(cid)

        db.insert_chunk(
            chunk_id=cid,
            text=text,
            embedding=emb
        )
    except Exception as e:
        print(f"Embedding failed at chunk {i}")
        failed_chunks.append(i)

print("Stored in vector DB.")
print("Total embeddings:", len(chunk_ids))
print("Failed:", len(failed_chunks))


# print("\nStep 5: Generating embeddings...")

# failed_chunks = []
# embeddings = []

# for i, text in enumerate(chunk_texts):
#     try:
#         emb = get_embedding(text)
#         embeddings.append(emb)

#     except Exception as e:
#         print(f"Embedding failed at chunk {i}")
#         failed_chunks.append(i)

# print("Total embeddings:", len(embeddings))
# print("Failed:", len(failed_chunks))



# -------------------------------
# 6. BUILD GRAPH (LLM → CYPHER → NEO4J)
# -------------------------------
print("\nStep 6: Building graph in Neo4j...")

# Important: use SAME chunk_texts as embeddings (cleaned)
build_graph_from_chunks(GraphRAG_PAPER_chunk_texts)

print("\nPipeline completed successfully.")





<frozen importlib._bootstrap>:228: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:228: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:228: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


Step 1: Extracting pages...
Total pages: 28

Step 2: Extracting blocks...
Total blocks: 975

Step 3: Chunking...
Total chunks: 71

Step 4: Building chunk texts...
Valid chunk_texts: 71
Creating schema...


c:\Users\Dell\OneDrive - Indian Institute of Science\Academics\Projects\hybrid-RAG\hybrid-rag-v1\lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


Storing embeddings...


c:\Users\Dell\OneDrive - Indian Institute of Science\Academics\Projects\hybrid-RAG\hybrid-rag-v1\lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'tvsmaznpiaoiprd01.openai.azure.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Stored in vector DB.
Total embeddings: 71
Failed: 0

Step 6: Building graph in Neo4j...
Processing chunk 1/71


c:\Users\Dell\OneDrive - Indian Institute of Science\Academics\Projects\hybrid-RAG\hybrid-rag-v1\lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'tvsmaznpiaoiprd02.cognitiveservices.azure.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Processing chunk 2/71
Processing chunk 3/71
Processing chunk 4/71
Processing chunk 5/71
Processing chunk 6/71
Processing chunk 7/71
Processing chunk 8/71
Processing chunk 9/71
Processing chunk 10/71
Processing chunk 11/71
Processing chunk 12/71
Processing chunk 13/71
Processing chunk 14/71
Processing chunk 15/71
Processing chunk 16/71
Processing chunk 17/71
Processing chunk 18/71
Processing chunk 19/71
Processing chunk 20/71
Processing chunk 21/71
Processing chunk 22/71
Processing chunk 23/71
Processing chunk 24/71
Processing chunk 25/71
Processing chunk 26/71
Processing chunk 27/71
Processing chunk 28/71
Processing chunk 29/71
Processing chunk 30/71
Processing chunk 31/71
Processing chunk 32/71
Processing chunk 33/71
Processing chunk 34/71
Processing chunk 35/71
Processing chunk 36/71
Processing chunk 37/71
Processing chunk 38/71
Processing chunk 39/71
Processing chunk 40/71
Processing chunk 41/71
Processing chunk 42/71
Processing chunk 43/71
Processing chunk 44/71
Processing chunk 45

In [ ]:
# %%

from services.retrieval.hybrid_engine import HybridQueryEngine

engine = HybridQueryEngine()

fl = True

while fl:
    q = input("Enter your query (or 'exit' to quit): ")
    if q.lower() == 'exit':
        print("Exiting...")
        print("-" * 50)
        fl = False
        continue
    result = engine.query(q)
    print("query=",q)
    print("result=",result['answer'])
    print("-" * 50)


